In [1]:
# import os
# import random #데이터 샘플링
# from collections import Counter # count 용도

import numpy as np
import pandas as pd

from tqdm import tqdm

# 결측치 확인
import missingno as msno

import warnings
warnings.filterwarnings('ignore')

# 시각화
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
plt.style.use('fivethirtyeight')

# 한글, 마이너스 깨짐 방지
from matplotlib import rc, font_manager, rcParams
font=font_manager.FontProperties(fname="c:/Windows/Fonts/malgun.ttf").get_name()
rc('font', family = font)
rcParams['axes.unicode_minus'] = False

# 지도
# from geopy import distance # 거리 계산
# import geopy.distance
import folium
from folium.plugins import HeatMap

# plotly
import ipywidgets as widgets
from ipywidgets import interact

# 이걸 설정하면 Multiple Output이 가능함
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import chart_studio.plotly as py 
import plotly.express as px
import cufflinks as cf 
cf.go_offline(connected=True)

import plotly.graph_objects as go

In [2]:
df= pd.read_csv('./dataset/restaurant_pre.csv')
df

,name,category,address,score,eval_cnt,review_cnt,lat,lng,distance
0,진미평양냉면,냉면,논현동 115-10,3.7,204,532,37.516107,127.036030,669.698432
1,대가방 본점,중화요리,논현동 99-7,3.3,86,101,37.521115,127.038526,1233.413552
2,세종한우 2호점,"육류,고기",논현동 216-10,3.0,27,42,37.510889,127.032509,235.981638
3,크래버대게나라 강남점,"게,대게",논현동 261,4.1,38,188,37.511590,127.036288,528.555530
4,카페써드,커피전문점,논현동 118-4,3.7,12,100,37.517362,127.039830,1020.061748
...,...,...,...,...,...,...,...,...,...
494,스윙부스,카페,논현동 151-17,5.0,4,15,37.511115,127.030901,122.943185
495,60계 서울논현점,치킨,논현동 183-2,3.6,8,24,37.506444,127.024751,800.969261
496,필스너하우스 청담점,"호프,요리주점",논현동 96-12,4.6,7,15,37.521158,127.037377,1180.985491
497,웨이크업커피 2호점,카페,논현동 86-4,5.0,1,3,37.515219,127.031794,367.396766


In [16]:
df['category'].unique()

array(['냉면', '중화요리', '육류,고기', '게,대게', '커피전문점', '한식', '곱창,막창', '족발,보쌈',
       '돈까스,우동', '조개', '초밥,롤', '일식', '카페', '삼계탕', '양식', '디저트카페', '장어',
       '중식', '일본식라면', '갈비', '퓨전일식', '샤브샤브', '일본식주점', '삼겹살', '해물,생선', '국수',
       '떡볶이', '닭요리', '호프,요리주점', '갤러리카페', '이탈리안', '패밀리레스토랑', '한정식', '회',
       '양꼬치', '참치회', '사철탕,영양탕', '북카페', '퓨전요리', '실내포장마차', '아구', '일식집',
       '햄버거', '제과,베이커리', '수제비', '설렁탕', '복어', '찌개,전골', '치킨', '매운탕,해물탕',
       '해장국', '태국음식', '피자', '순대', '술집', '패스트푸드', '와인바', '칵테일바', '베트남음식',
       '애견카페', '국밥', '다방', '불고기,두루치기', '감자탕', '샌드위치', '동남아음식', '분식',
       '음식점', '죽', '도시락', '굴,전복'], dtype=object)

+ 중식 중화요리
+ 한식 한정식
+ 카페 커피전문점 디저트카페 갤러리카페 북카페 애견카페
+ 햄버거 패스트푸드
+ 일식 일본식라면
+ 양식 패밀리레스토랑
+ 술집 일본식주점 호프,요리주점 실내포장마차 와인바 칵테일바

In [35]:
for i in df.index:
    if df.loc[i, 'category']=='중화요리':
        df.loc[i, 'category']='중식'
    elif df.loc[i, 'category']=='한정식':
        df.loc[i, 'category']='한식'
    elif (df.loc[i, 'category']=='커피전문점') or (df.loc[i, 'category']=='디저트카페') or (df.loc[i, 'category']=='갤러리카페') or (df.loc[i, 'category']=='북카페') or (df.loc[i, 'category']=='애견카페') or (df.loc[i, 'category']=='다방'):
        df.loc[i, 'category']='카페'
    elif (df.loc[i, 'category']=='패스트푸드'):
        df.loc[i, 'category']='햄버거'
    elif (df.loc[i, 'category']=='일본식주점') or (df.loc[i, 'category']=='호프,요리주점') or (df.loc[i, 'category']=='실내포장마차') or (df.loc[i, 'category']=='와인바') or (df.loc[i, 'category']=='칵테일바'):
        df.loc[i, 'category']='술집'
    elif df.loc[i, 'category']=='패밀리레스토랑':
        df.loc[i, 'category']='양식'
    elif df.loc[i, 'category']=='음식점':
        df.loc[i, 'category']='퓨전요리'

In [36]:
df['category'].unique()

array(['냉면', '중식', '육류,고기', '게,대게', '카페', '한식', '곱창,막창', '족발,보쌈',
       '돈까스,우동', '조개', '초밥,롤', '일식', '삼계탕', '양식', '장어', '일본식라면', '갈비',
       '퓨전일식', '샤브샤브', '술집', '삼겹살', '해물,생선', '국수', '떡볶이', '닭요리', '이탈리안',
       '회', '양꼬치', '참치회', '사철탕,영양탕', '퓨전요리', '아구', '일식집', '햄버거',
       '제과,베이커리', '수제비', '설렁탕', '복어', '찌개,전골', '치킨', '매운탕,해물탕', '해장국',
       '태국음식', '피자', '순대', '베트남음식', '국밥', '불고기,두루치기', '감자탕', '샌드위치',
       '동남아음식', '분식', '죽', '도시락', '굴,전복'], dtype=object)

In [34]:
df[df['category']=='퓨전요리']

,name,category,address,score,eval_cnt,review_cnt,lat,lng,distance
81,우기식당 육지점,퓨전요리,논현동 144-8,4.3,20,18,37.508991,127.023439,700.941448
116,롤리폴리꼬또,퓨전요리,논현동 269-10,3.3,21,129,37.511234,127.042322,1061.998319


In [7]:
import chart_studio
chart_studio.tools.set_credentials_file(username='mingkipark', api_key='********')
from chart_studio.plotly import plot, iplot

In [39]:
import plotly.graph_objects as go


def aca_trace(figure, m, t, c='gray', o=.2):
    aca_geo = (37.5121248, 127.030334)
    lat100m = 0.000899
    lng100m = 0.001134
    lat0 = aca_geo[0] - lat100m * m
    lat1 = aca_geo[0] + lat100m * m
    lng0 = aca_geo[1] - lng100m * m
    lng1 = aca_geo[1] + lng100m * m
    figure.add_trace(go.Scattermapbox(
        mode="markers+lines", 
        showlegend=False,
        opacity=o,
        lat=[lat0, lat0, lat1, lat1, lat0],
        lon=[lng0, lng1, lng1, lng0, lng0],
        marker={'color': c},
        hovertemplate='도보 {}분'.format(t),
        name='넥스트허브 근처 {}00m'.format(m)))

fig = px.scatter_mapbox(df, lat="lat", lon="lng", zoom=14,
                        size='score', size_max=10, 
                        hover_name='name',
                        color='category',
                        mapbox_style='carto-positron', width=1000, height=1000,
                        center={'lat': df.lat.mean(), 'lon': df.lng.mean()})
aca_trace(fig, 1, 1, c='black', o=.5)
aca_trace(fig, 3, 5, o=.5)
aca_trace(fig, 7, 10, o=.3)
fig.show()

In [38]:
# plot(fig, filename = 'nexthub_JMT', auto_open=True)

'https://plotly.com/~mingkipark/27/'

In [40]:
df.head()

,name,category,address,score,eval_cnt,review_cnt,lat,lng,distance
0,진미평양냉면,냉면,논현동 115-10,3.7,204,532,37.516107,127.036030,669.698432
1,대가방 본점,중식,논현동 99-7,3.3,86,101,37.521115,127.038526,1233.413552
2,세종한우 2호점,"육류,고기",논현동 216-10,3.0,27,42,37.510889,127.032509,235.981638
3,크래버대게나라 강남점,"게,대게",논현동 261,4.1,38,188,37.511590,127.036288,528.555530
4,카페써드,카페,논현동 118-4,3.7,12,100,37.517362,127.039830,1020.061748


In [41]:
hovertemplate = []
for num in df.index:
    name = df.iloc[num, 0]
    category = df.iloc[num, 1]
    address = df.iloc[num, 2]
    score = df.iloc[num, 3]
    eval_cnt = df.iloc[num, 4]
    review_cnt = df.iloc[num, 5]
    
    hovertemplate.append(f"{name} <br>========<br>카테고리 : {category}<br>주소 : {address}<br>평점 : {score}<br>평가 수 : {eval_cnt}<br>리뷰 수 : {review_cnt}")

In [50]:
df.rename(columns={'score':'평점', 'eval_cnt':'평가 수', 'review_cnt':'리뷰 수'}, inplace=True)
df.head()

,name,category,address,평점,평가 수,리뷰 수,lat,lng,distance
0,진미평양냉면,냉면,논현동 115-10,3.7,204,532,37.516107,127.036030,669.698432
1,대가방 본점,중식,논현동 99-7,3.3,86,101,37.521115,127.038526,1233.413552
2,세종한우 2호점,"육류,고기",논현동 216-10,3.0,27,42,37.510889,127.032509,235.981638
3,크래버대게나라 강남점,"게,대게",논현동 261,4.1,38,188,37.511590,127.036288,528.555530
4,카페써드,카페,논현동 118-4,3.7,12,100,37.517362,127.039830,1020.061748


In [51]:
import plotly.graph_objects as go


def aca_trace(figure, m, t, c='gray', o=.2):
    aca_geo = (37.5121248, 127.030334)
    lat100m = 0.000899
    lng100m = 0.001134
    lat0 = aca_geo[0] - lat100m * m
    lat1 = aca_geo[0] + lat100m * m
    lng0 = aca_geo[1] - lng100m * m
    lng1 = aca_geo[1] + lng100m * m
    figure.add_trace(go.Scattermapbox(
        mode="markers+lines", 
        showlegend=False,
        opacity=o,
        lat=[lat0, lat0, lat1, lat1, lat0],
        lon=[lng0, lng1, lng1, lng0, lng0],
        marker={'color': c},
        hovertemplate='도보 {}분'.format(t),
        name='넥스트허브 근처 {}00m'.format(m)))

fig = px.scatter_mapbox(df, lat="lat", lon="lng", zoom=14,
#                         size='score', size_max=10, 
                        hover_name='name',
                        hover_data=['category', 'address', '평점', '평가 수', '리뷰 수'],
                        color='category',
                        mapbox_style='carto-positron', width=1000, height=1000,
                        center={'lat': df.lat.mean(), 'lon': df.lng.mean()})
aca_trace(fig, 1, 1, c='black', o=.5)
aca_trace(fig, 3, 5, o=.5)
aca_trace(fig, 7, 10, o=.3)
fig.show()

In [52]:
# plot(fig, filename = 'nexthub_JMT', auto_open=True)

'https://plotly.com/~mingkipark/27/'